[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/coregionalization_toy.ipynb)

# Coregionalized regression (ICM) with `gpc`

Native-twin tutorial for [coregionalization_toy](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/coregionalization_toy.md), using the `gpc` façade (CIP-0038 Phase D). Related GPy reading: [`GPy/coregionalized_regression_tutorial.ipynb`](../GPy/coregionalized_regression_tutorial.ipynb) (longer kernel-selection narrative; different synthetic data).

Requires `gpc>=0.1.0a8` (also on `0.1.0a7`).

```python
m = gpc.GPCoregionalizedRegression(X_list, Y_list, kernel=gpc.kern.RBF(1), rank=1)
m.optimize()
mu, var = m.predict_output(Xstar, output=0)
```

**Notes vs GPy**

* Stack via `gpc.util.multioutput.build_XY` / `ICM` / `LCM`.
* Default adds per-output `IndexWhite`; spherical `likelihood.variance` is frozen tiny (CIP-000D / CIP-0027).
* Predictions use `predict_output` (GPy often stamps an output index on \(X\) and passes `Y_metadata`).
* `objective()` is **−log-likelihood**.
* Sparse / MixedNoise multi-output object graph still deferred.


In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Two noisy sinusoids

Synthesise two outputs of different length (native `coregionalization_toy` / GPy example sizes: \(N_0=50\) on \([0,8]\), \(N_1=30\) on \([0,5]\), second output offset by \(+2\)).


In [ ]:
rng = np.random.default_rng(0)
N0, N1 = 50, 30
X0 = np.linspace(0.0, 8.0, N0)[:, None]
X1 = np.linspace(0.0, 5.0, N1)[:, None]
Y0 = np.sin(X0) + 0.1 * rng.normal(size=X0.shape)
Y1 = np.sin(X1) + 2.0 + 0.1 * rng.normal(size=X1.shape)


## Building an ICM kernel

Intrinsic coregionalization is a spatial kernel tensored with a coregionalization matrix \(B = WW^\top + \mathrm{diag}(\kappa)\). In `gpc` you can multiply the factors by hand or call the GPy-shaped helper.


In [ ]:
K = gpc.kern.RBF(1, lengthscale=1.0)
B = gpc.kern.Coregionalize(1, output_dim=2, rank=1)
manual = K * B
icm = gpc.util.multioutput.ICM(1, 2, gpc.kern.RBF(1, lengthscale=1.0), W_rank=1)
print("manual:", manual)
print("ICM helper:", icm)
print("parts:", icm.to_parts())


## ICM model

Pass a **spatial** kernel into `GPCoregionalizedRegression`; the façade builds ICM and (by default) adds `IndexWhite` for per-output noise.


In [ ]:
m = gpc.GPCoregionalizedRegression(
    [X0, X1], [Y0, Y1], kernel=gpc.kern.RBF(1, lengthscale=1.0), rank=1
)
print(m)
coreg_paths = [p for p in m.param_paths() if "coregionalize" in p or "indexwhite" in p]
print("coreg / indexwhite paths:", coreg_paths)
m.optimize(iters=40, messages=False)
print("objective", m.objective())
for p in coreg_paths:
    print(f"  {p} = {m.get_by_path(p)}")


## Predictions per output

`predict_output` stamps the output index for you. Under the hood this is the same extended input space GPy exposes explicitly (`[x, output_id]` plus noise metadata).


In [ ]:
Xs = np.linspace(0.0, 8.0, 120)[:, None]
# Optional: see the stamped design matrix GPy would pass to predict.
Xs_out1 = gpc.util.multioutput.stamp_output_index(
    np.hstack([Xs[:3], np.zeros((3, 1))]), 1
)
print("stamped X for output 1 (first 3):\n", Xs_out1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, Xi, Yi, out, title in (
    (axes[0], X0, Y0, 0, "output 0"),
    (axes[1], X1, Y1, 1, "output 1"),
):
    mu, var = m.predict_output(Xs, output=out)
    sd = np.sqrt(np.maximum(var, 0.0))
    ax.plot(Xi, Yi, "kx", ms=4, alpha=0.7)
    ax.plot(Xs, mu, "C0-")
    ax.fill_between(
        Xs[:, 0], (mu - 2 * sd)[:, 0], (mu + 2 * sd)[:, 0], color="C0", alpha=0.2
    )
    ax.set_title(title)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("ICM + IndexWhite")
fig.tight_layout()
plt.show()


## Why share? Independent GPs vs ICM

Two separate regressions cannot borrow strength where one output has no data. On this toy, output 1 only observes \([0,5]\); the ICM mean on \((5,8]\) is informed by output 0.


In [ ]:
m0 = gpc.models.GPRegression(X0, Y0, kernel=gpc.kern.RBF(1))
m1 = gpc.models.GPRegression(X1, Y1, kernel=gpc.kern.RBF(1))
m0.optimize(iters=40, messages=False)
m1.optimize(iters=40, messages=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, Xi, Yi, out, mind, title in (
    (axes[0], X0, Y0, 0, m0, "output 0"),
    (axes[1], X1, Y1, 1, m1, "output 1"),
):
    mu_c, var_c = m.predict_output(Xs, output=out)
    # Independent models: predict only on their own input dim.
    mu_i, var_i = mind.predict(Xs)
    sd_c = np.sqrt(np.maximum(var_c, 0.0))
    sd_i = np.sqrt(np.maximum(var_i, 0.0))
    ax.plot(Xi, Yi, "kx", ms=4, alpha=0.7)
    ax.plot(Xs, mu_c, "C0-", label="ICM")
    ax.fill_between(
        Xs[:, 0], (mu_c - 2 * sd_c)[:, 0], (mu_c + 2 * sd_c)[:, 0], color="C0", alpha=0.15
    )
    ax.plot(Xs, mu_i, "C1--", label="independent")
    ax.fill_between(
        Xs[:, 0], (mu_i - 2 * sd_i)[:, 0], (mu_i + 2 * sd_i)[:, 0], color="C1", alpha=0.1
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.legend(fontsize=8)
axes[0].set_ylabel("y")
fig.suptitle("ICM shares structure; independent GPs do not")
fig.tight_layout()
plt.show()


## Matérn ICM variant

Same construction with a Matérn-3/2 spatial factor (common in the GPy tutorial).


In [ ]:
m_mat = gpc.GPCoregionalizedRegression(
    [X0, X1],
    [Y0, Y1],
    kernel=gpc.kern.Matern32(1, lengthscale=1.0),
    rank=1,
)
m_mat.optimize(iters=40, messages=False)
print(m_mat)
print("Matérn ICM objective", m_mat.objective())


## LCM: sum of ICMs

Linear coregionalization is a sum of ICM terms. Use it when different spatial structures (here RBF + Linear) should share information with separate \(B\) matrices. For non-zero-mean / trend-dominated data the GPy tutorial often starts from Bias + Linear; this sinusoid toy only needs a light illustration.


In [ ]:
lcm = gpc.util.multioutput.LCM(
    1, 2, [gpc.kern.RBF(1), gpc.kern.Linear(1)], W_rank=1
)
print("LCM:", lcm)
m_lcm = gpc.GPCoregionalizedRegression([X0, X1], [Y0, Y1], kernel=lcm, indexwhite=True)
m_lcm.optimize(iters=25, messages=False)
print("LCM objective", m_lcm.objective())
